# Combine physical store returns data and export Parquet file

### 1) Create a Spark session

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
  .appName("Physical Store Returns Processing")\
  .master("yarn")\
  .enableHiveSupport()\
  .getOrCreate()

In [ ]:
spark

### 2) Load store returns data from a CSV file into a DataFrame

In [ ]:
# Read data from CSV files in Google Cloud Storage and populate a DataFrame
stores_dataframe = spark.read.csv("gs://sureskills-lab-dev/DAC2M4L1/store_returns_input/store_returns_*.csv", header=True, inferSchema=True)


In [ ]:
# Print the DataFrame schema to review data types and field names
stores_dataframe.printSchema()

In [ ]:
display(stores_dataframe.show(3,truncate=False))

In [ ]:
# Create a temporary view to reference the DataFrame in Spark SQL
# Note the name of the view: store_returns
stores_dataframe.createOrReplaceTempView("store_returns")

In [ ]:
# Display the store_returns data using Spark SQL and the temporary view store_returns created in the prior step
# Note that the temporary view is referenced as a table in the SELECT statement
spark.sql("select * from store_returns").show()

### 3) Load store names from a CSV file

In [ ]:
# Read data from CSV files in Google Cloud Storage (GCS) and populate a DataFrame
stores_dataframe = spark.read.csv("gs://sureskills-lab-dev/DAC2M4L1/store_returns_input/stores_*.csv", header=True, inferSchema=True)


In [ ]:
# Print schema
stores_dataframe.printSchema()

In [ ]:
stores_dataframe.createOrReplaceTempView("stores")

In [ ]:
spark.sql("select * from stores").show()

### 4) Add street address to returns data and export it as Parquet file

In [ ]:
# The Spark SQL query joins the store returns data with store information data, populates a new DataFrame and displays it 
dataframe_to_export = spark.sql('''
                                    select  
                                      sr.order_id, 
                                      sr.rma_id,  
                                      sr.return_status,         
                                      sr.status_date,  
                                      sr.product_id,  
                                      sr.quantity_returned,  
                                      sr.store_id,  
                                      s.street_address  
                                    from store_returns sr  
                                    inner join stores as s  
                                    on sr.store_id = s.store_id    
                                ''')
dataframe_to_export.show()

In [ ]:
# Find the Google Cloud Project ID.
PROJECT_ID=!gcloud info --format='value(config.project)'

# Export dataframe as a Parquet File - overwrite the file if it exists
# Note that a long line is broken off using the backlash (\)
# The Google Cloud Storage bucket name in this lab is the name of the project id
# We are using the project_id value obtained in the prior step in the gs:// path
dataframe_to_export.write\
  .mode("overwrite")\
  .save("gs://" + PROJECT_ID[0] + "/store_returns_output/store_returns.parquet")

#### 4.2. Modify a query

In [ ]:
# Modify the query below so that the field name displayed is store_address instead of street_address
dataframe_to_export = spark.sql('''
                                    select  
                                      sr.order_id, 
                                      sr.rma_id,  
                                      sr.return_status,         
                                      sr.status_date,  
                                      sr.product_id,  
                                      sr.quantity_returned,  
                                      sr.store_id,  
                                      s.street_address 
                                    from store_returns sr  
                                    inner join stores as s  
                                    on sr.store_id = s.store_id    
                                ''')
dataframe_to_export.show()

# Hint: the answer is in the next cell

In [ ]:
# This query modifies the field name to display store_address instead of street_address
dataframe_to_export = spark.sql('''
                                    select  
                                      sr.order_id, 
                                      sr.rma_id,  
                                      sr.return_status,         
                                      sr.status_date,  
                                      sr.product_id,  
                                      sr.quantity_returned,  
                                      sr.store_id,  
                                      s.street_address as store_address
                                    from store_returns sr  
                                    inner join stores as s  
                                    on sr.store_id = s.store_id    
                                ''')
dataframe_to_export.show()